In [2]:
import numpy as np

In [3]:
file_path = '../data/raw/aug_train.csv'

# Đọc tên cột
try:
    with open(file_path, 'r') as f:
        header_line = f.readline().strip()
        cols = header_line.split(',') 
except FileNotFoundError:
    print("Không tìm thấy file.")
    cols = []

# Tải dữ liệu thành mảng 2D (N, M)
data = np.genfromtxt(
    file_path,
    delimiter=',',
    skip_header=1,
    dtype=str
)

# In những kết quả đơn giản
print('Column names:')
for col_name in cols:
    print(f"- {col_name}")
print(f'\nShape: {data.shape}')

Column names:
- enrollee_id
- city
- city_development_index
- gender
- relevent_experience
- enrolled_university
- education_level
- major_discipline
- experience
- company_size
- company_type
- last_new_job
- training_hours
- target

Shape: (19158, 14)


In [4]:
print('First 10 rows:\n')
for i in range(10):
    print(data[i])
    print()


First 10 rows:

['8949' 'city_103' '0.92' 'Male' 'Has relevent experience' 'no_enrollment'
 'Graduate' 'STEM' '>20' '' '' '1' '36' '1.0']

['29725' 'city_40' '0.7759999999999999' 'Male' 'No relevent experience'
 'no_enrollment' 'Graduate' 'STEM' '15' '50-99' 'Pvt Ltd' '>4' '47' '0.0']

['11561' 'city_21' '0.624' '' 'No relevent experience' 'Full time course'
 'Graduate' 'STEM' '5' '' '' 'never' '83' '0.0']

['33241' 'city_115' '0.789' '' 'No relevent experience' '' 'Graduate'
 'Business Degree' '<1' '' 'Pvt Ltd' 'never' '52' '1.0']

['666' 'city_162' '0.767' 'Male' 'Has relevent experience' 'no_enrollment'
 'Masters' 'STEM' '>20' '50-99' 'Funded Startup' '4' '8' '0.0']

['21651' 'city_176' '0.764' '' 'Has relevent experience'
 'Part time course' 'Graduate' 'STEM' '11' '' '' '1' '24' '1.0']

['28806' 'city_160' '0.92' 'Male' 'Has relevent experience'
 'no_enrollment' 'High School' '' '5' '50-99' 'Funded Startup' '1' '24'
 '0.0']

['402' 'city_46' '0.762' 'Male' 'Has relevent experience'

In [5]:
# Đếm các dữ liệu bị thiếu của từng cột

missing = {}

for col_idx, col_name in enumerate(cols):
    col_data = data[:, col_idx]
    miss = np.sum(
        (col_data == "") |
        (col_data == " ")
    )
    missing[col_name] = miss

print("Missing values per column:")
for k, v in missing.items():
    print(f"- {k}: {v}")

    

Missing values per column:
- enrollee_id: 0
- city: 0
- city_development_index: 0
- gender: 4508
- relevent_experience: 0
- enrolled_university: 386
- education_level: 460
- major_discipline: 2813
- experience: 65
- company_size: 5938
- company_type: 6140
- last_new_job: 423
- training_hours: 0
- target: 0


In [6]:
# Xác định loại dữ liệu của từng cột

def detect_type(col):
    count_num = 0
    for v in col:
        try:
            float(v)
            count_num += 1
        except:
            pass
    if count_num == len(col):
        return "Numeric"
    elif count_num >= 0.7 * len(col):
        return "Mostly Numeric"
    else:
        return "Categorical"

print("Column types:")
for i, name in enumerate(cols):
    print(f"- {name}: {detect_type(data[:, i])}")


Column types:
- enrollee_id: Numeric
- city: Categorical
- city_development_index: Numeric
- gender: Categorical
- relevent_experience: Categorical
- enrolled_university: Categorical
- education_level: Categorical
- major_discipline: Categorical
- experience: Mostly Numeric
- company_size: Categorical
- company_type: Categorical
- last_new_job: Categorical
- training_hours: Numeric
- target: Numeric


In [7]:
# Thống kê mô tả cho cột numeric

numeric_stats = {}

for idx, name in enumerate(cols):
    try:
        arr = data[:, idx].astype(float)
        numeric_stats[name] = {
            "min": float(np.min(arr)),
            "max": float(np.max(arr)),
            "mean": float(np.mean(arr)),
            "std": float(np.std(arr))
        }
    except:
        pass

print("Numeric columns summary:")
for name, stats in numeric_stats.items():
    print(f"\n- {name}:")
    for k, v in stats.items():
        print(f"  {k}: {v}")


Numeric columns summary:

- enrollee_id:
  min: 1.0
  max: 33380.0
  mean: 16875.358179350664
  std: 9616.041615595486

- city_development_index:
  min: 0.44799999999999995
  max: 0.9490000000000001
  mean: 0.8288480008351603
  std: 0.12335853722992858

- training_hours:
  min: 1.0
  max: 336.0
  mean: 65.36689633573442
  std: 60.05689445297729

- target:
  min: 0.0
  max: 1.0
  mean: 0.24934753105752167
  std: 0.43263534276921933


In [8]:
from collections import Counter

categorical_cols = [c for c in cols if c not in numeric_stats]

for name in categorical_cols:
    idx = cols.index(name)
    vals = data[:, idx]
    counts = Counter(vals)

    print(f"\nTop values in column '{name}':")
    for v, c in counts.most_common(10):
        print(f"  {v}: {c}")



Top values in column 'city':
  city_103: 4355
  city_21: 2702
  city_16: 1533
  city_114: 1336
  city_160: 845
  city_136: 586
  city_67: 431
  city_75: 305
  city_102: 304
  city_104: 301

Top values in column 'gender':
  Male: 13221
  : 4508
  Female: 1238
  Other: 191

Top values in column 'relevent_experience':
  Has relevent experience: 13792
  No relevent experience: 5366

Top values in column 'enrolled_university':
  no_enrollment: 13817
  Full time course: 3757
  Part time course: 1198
  : 386

Top values in column 'education_level':
  Graduate: 11598
  Masters: 4361
  High School: 2017
  : 460
  Phd: 414
  Primary School: 308

Top values in column 'major_discipline':
  STEM: 14492
  : 2813
  Humanities: 669
  Other: 381
  Business Degree: 327
  Arts: 253
  No Major: 223

Top values in column 'experience':
  >20: 3286
  5: 1430
  4: 1403
  3: 1354
  6: 1216
  2: 1127
  7: 1028
  10: 985
  9: 980
  8: 802

Top values in column 'company_size':
  : 5938
  50-99: 3083
  100-500: 2

In [9]:
print("Outlier detection (|z| > 3):\n")

for name, stats in numeric_stats.items():
    idx = cols.index(name)
    arr = data[:, idx].astype(float)

    z = (arr - stats["mean"]) / stats["std"]
    outliers = np.sum(np.abs(z) > 3)

    print(f"{name}: {outliers} outliers")


Outlier detection (|z| > 3):

enrollee_id: 0 outliers
city_development_index: 17 outliers
training_hours: 450 outliers
target: 0 outliers


In [10]:
# build numeric matrix
numeric_cols = list(numeric_stats.keys())
numeric_matrix = np.column_stack([data[:, cols.index(c)].astype(float) for c in numeric_cols])

corr = np.corrcoef(numeric_matrix, rowvar=False)

print("Correlation matrix (numeric features):")
print("Columns:", numeric_cols)
print(corr)


: 

In [ ]:
print("""
✅ Data Exploration Completed!

Included:
- Data loading
- Missing value check
- Type detection
- Numeric statistics
- Categorical value analysis
- Outlier detection
- Correlation matrix
""")
